In [1]:
# ==========================================
# CELL 1: SETUP API & PATH FILE
# ==========================================
import os
import time
import pandas as pd
from groq import Groq

# Setup API Key Groq
api_key_groq = os.getenv("GROQ_API_KEY")

# Cek apakah terbaca (Print 8 karakter pertama saja untuk keamanan)
print(f"🔑 Key yang terbaca oleh sistem: {api_key_groq[:8]}...") 

# Jika hasil print memunculkan 'gsk_1yqR...' (tanpa tanda kutip), berarti SUKSES!
client = Groq(api_key=api_key_groq)
print("✅ Client Groq berhasil diinisialisasi!")

# Setup Path (Sesuaikan dengan struktur folder repo exigen-smart-maintenance Anda)
path_df_aset = "../../data/master_aset_enriched.xlsx" 
path_df_perbaikan = "../../data/aset_komplain_enriched.xlsx"
path_output = "../../data/dataset_keluhan_ntg.csv"

print("✅ Cell 1 Selesai: Library ter-import dan API Key sudah diset.")

🔑 Key yang terbaca oleh sistem: gsk_1yqR...
✅ Client Groq berhasil diinisialisasi!
✅ Cell 1 Selesai: Library ter-import dan API Key sudah diset.


In [ ]:
# ==========================================
# CELL 2: LOAD DATA & AMBIL ATURAN (CONSTRAINTS)
# ==========================================
print("Membaca dataset asli NTG...")
df_aset = pd.read_excel(path_df_aset)
df_perbaikan = pd.read_excel(path_df_perbaikan)

# Mengambil daftar Kunci Jawaban (Target Y) agar AI tidak ngarang
list_severity = df_perbaikan['Severity'].dropna().unique().tolist()
list_penyebab = df_perbaikan['Penyebab'].dropna().unique().tolist()
list_jenis_kerusakan = df_perbaikan['Jenis Kerusakan'].dropna().unique().tolist() 

# Mengambil Kombinasi Aset Nyata (Feature X)
kombinasi_valid = df_aset[['Kategori', 'Sub Kategori', 'Tipe', 'Merek', 
                           'Lokasi Gedung', 'Lokasi Lantai', 'Lokasi Zona']].dropna().drop_duplicates()

# UNTUK TESTING: Kita ambil 3 kombinasi acak saja dulu agar prosesnya cuma hitungan detik
kombinasi_sample = kombinasi_valid.sample(3) 

print(f"✅ Cell 2 Selesai: Berhasil mengekstrak {len(kombinasi_sample)} kombinasi mesin untuk di-testing.")
print(f"🔸 Batas Severity yang diizinkan: {list_severity[:3]}...")

Membaca dataset asli NTG...
✅ Cell 2 Selesai: Berhasil mengekstrak 3 kombinasi mesin untuk di-testing.
🔸 Batas Severity yang diizinkan: ['Fatal', 'Ringan', 'Sedang']...


In [ ]:
# ==========================================
# CELL 2.1: VALIDASI DATA UNIK (MAKESURE DATA BENAR)
# ==========================================

print("🔍 === RINGKASAN DATA UNIK UNTUK PROMPT AI === 🔍\n")

# 1. Validasi Kolom Target (dari df_perbaikan)
print("--- [TARGET Y / KUNCI JAWABAN] ---")
print(f"🔹 Severity ({len(list_severity)}): {list_severity}")
print(f"🔹 Penyebab ({len(list_penyebab)}): {list_penyebab[:10]}... (dan seterusnya)")
print(f"🔹 Tindakan ({len(list_jenis_kerusakan)}): {list_jenis_kerusakan[:10]}... (dan seterusnya)")

# 2. Validasi Identitas & Lokasi (dari df_aset)
print("\n--- [IDENTITAS ASET & LOKASI] ---")
cols_aset = ['Kategori', 'Sub Kategori', 'Tipe', 'Merek', 'Lokasi Gedung', 'Lokasi Lantai', 'Lokasi Zona']

for col in cols_aset:
    unique_vals = df_aset[col].dropna().unique().tolist()
    print(f"📍 {col} ({len(unique_vals)} unik): {unique_vals[:15]}") # Tampilkan 15 contoh pertama

# 3. Validasi Kombinasi yang Terpilih untuk Testing
print("\n--- [CONTOH KOMBINASI UNTUK TESTING GROQ] ---")
display(kombinasi_sample)

🔍 === RINGKASAN DATA UNIK UNTUK PROMPT AI === 🔍

--- [TARGET Y / KUNCI JAWABAN] ---
🔹 Severity (4): ['Fatal', 'Ringan', 'Sedang', 'Berat']
🔹 Penyebab (12): ['Kelembaban tinggi', 'Human error', 'Overload', 'Kurang perawatan', 'Tegangan tidak stabil', 'Faktor lingkungan', 'Debu/kotoran', 'Aus normal', 'Usia pakai', 'Kualitas material']... (dan seterusnya)
🔹 Tindakan (20): ['Retak/pecah', 'Overheat', 'Remote tidak berfungsi', 'Getaran berlebihan', 'Korsleting', 'Aus/abrasi', 'Tidak berfungsi total', 'Kebocoran', 'Sensor error', 'Aliran lemah']... (dan seterusnya)

--- [IDENTITAS ASET & LOKASI] ---
📍 Kategori (15 unik): ['Mechanical', 'Ventilasi Sistem', 'Electrical', 'Sistem Pemadam Kebakaran', 'Sistem Telekomunikasi Gedung', 'Sistem Proteksi Kebakaran Aktif', 'Security Sistem', 'Civil', 'Plumbing', 'Distribusi Air', 'Sistem Transportasi Gedung', 'Pencatatan Meter', 'Arsitektur', 'Sistem Energi', 'Latihan Balakar']
📍 Sub Kategori (51 unik): ['Tata Udara', 'Sistem Sirkulasi Udara', 'Contro

,Kategori,Sub Kategori,Tipe,Merek,Lokasi Gedung,Lokasi Lantai,Lokasi Zona
3568,Plumbing,Sanitari Sistem,Wastafel,Ina,Gedung B,4,Selatan
31821,Mechanical,Tata Udara,AC Split,Daikin,Gedung A,2,Timur
33284,Electrical,Lampu Penerangan,Lampu LED Downlight,Generic,Gedung A,14,Barat


In [4]:
# ==========================================
# CELL 3: LOOPING GENERATE DATASET VIA GROQ (REVISI DINAMIS)
# ==========================================
import time

print(f"Mulai generate data via LLaMA 3...")

csv_final = "teks_keluhan_awam|teks_laporan_teknisi|kategori_aset|severity|root_cause|jenis_kerusakan|biaya_perbaikan\n"

for index, row in kombinasi_sample.iterrows():
    kategori, tipe, merek = row['Kategori'], row['Tipe'], row['Merek']
    gedung, lantai, zona = row['Lokasi Gedung'], row['Lokasi Lantai'], row['Lokasi Zona']
    
    # 🌟 FITUR BARU: FILTERING KUNCI JAWABAN SPESIFIK PER KATEGORI 🌟
    # Kita cari di data histori teknisi, kerusakan apa saja yang PERNAH terjadi khusus di kategori ini
    df_spesifik = df_perbaikan[df_perbaikan['Kategori'] == kategori]
    
    # Jika datanya ada, ambil list spesifik. Jika kosong (aset baru), pakai list global sebagai cadangan
    if not df_spesifik.empty:
        list_penyebab_spesifik = df_spesifik['Penyebab'].dropna().unique().tolist()
        list_kerusakan_spesifik = df_spesifik['Jenis Kerusakan'].dropna().unique().tolist()
    else:
        list_penyebab_spesifik = list_penyebab # List global dari Cell 2
        list_kerusakan_spesifik = list_jenis_kerusakan # List global dari Cell 2
    
    print(f"🔄 Requesting: {kategori} - {merek} ({gedung} Lt {lantai})...")
    
    prompt = f"""
    Anda adalah AI pembuat dataset Machine Learning.
    Buatkan 15 baris data dummy laporan komplain kerusakan untuk spesifikasi aset ini:
    - Kategori: {kategori}
    - Tipe: {tipe}
    - Merek: {merek}
    - Lokasi: {gedung} Lantai {lantai} Zona {zona}

    Format WAJIB: CSV murni dengan pemisah '|'.
    Kolom: teks_keluhan_awam|teks_laporan_teknisi|kategori_aset|severity|root_cause|jenis_kerusakan|biaya_perbaikan

    ATURAN SANGAT KETAT:
    1. Selipkan '{merek}', '{tipe}', dan lokasi ('{gedung}', 'Lantai {lantai}', 'Zona {zona}') secara natural/acak ke dalam kalimat 'teks_keluhan_awam' atau 'teks_laporan_teknisi'. Buat bahasa awam yang panik.
    2. Kolom 'kategori_aset' WAJIB diisi persis: "{kategori}".
    3. Kolom 'severity' WAJIB dipilih persis dari list ini: {list_severity}
    4. Kolom 'root_cause' WAJIB dipilih persis HANYA dari list ini: {list_penyebab_spesifik}
    5. Kolom 'jenis_kerusakan' WAJIB dipilih persis HANYA dari list ini: {list_kerusakan_spesifik}
    6. Kolom 'biaya_perbaikan' diisi angka wajar dalam Rupiah (misal: 150000).
    7. OUTPUT MURNI CSV. JANGAN ada markdown (```csv). JANGAN keluarkan header di baris pertama.
    """
    
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile", 
            messages=[
                {"role": "system", "content": "You output ONLY raw valid CSV text without headers and without markdown blocks."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.7,
            max_tokens=3000
        )
        
        hasil_text = response.choices[0].message.content.strip()
        hasil_text = hasil_text.replace("```csv", "").replace("```", "").strip()
        
        if hasil_text.lower().startswith("teks_keluhan"):
            hasil_text = "\n".join(hasil_text.split("\n")[1:])
            
        csv_final += hasil_text + "\n"
        time.sleep(3) 
        
    except Exception as e:
        print(f"❌ Gagal pada kombinasi {kategori}-{merek}. Error: {e}")

print("✅ Cell 3 Selesai: Teks CSV berhasil ditampung di memori.")

Mulai generate data via LLaMA 3...
🔄 Requesting: Plumbing - Ina (Gedung B Lt 4)...
🔄 Requesting: Mechanical - Daikin (Gedung A Lt 2)...
🔄 Requesting: Electrical - Generic (Gedung A Lt 14)...
✅ Cell 3 Selesai: Teks CSV berhasil ditampung di memori.


In [5]:
# ==========================================
# CELL 4: SIMPAN DAN BACA HASILNYA
# ==========================================
# 1. Tulis string gabungan tadi ke dalam file
with open(path_output, "w", encoding="utf-8") as f:
    f.write(csv_final.strip())

print(f"File berhasil disimpan di: {path_output}")

# 2. Coba baca menggunakan Pandas untuk memastikan formatnya valid
try:
    df_hasil = pd.read_csv(path_output, sep='|', on_bad_lines='skip')
    print(f"\n✅ UJI COBA SUKSES! Total Data Terkumpul: {df_hasil.shape[0]} baris.")
    display(df_hasil.head())
except Exception as e:
    print(f"❌ Gagal membaca CSV. Pemisah kolom mungkin berantakan. Error: {e}")# ==========================================
# CELL 4: SIMPAN DAN BACA HASILNYA
# ==========================================
# 1. Tulis string gabungan tadi ke dalam file
with open(path_output, "w", encoding="utf-8") as f:
    f.write(csv_final.strip())

print(f"File berhasil disimpan di: {path_output}")

# 2. Coba baca menggunakan Pandas untuk memastikan formatnya valid
try:
    df_hasil = pd.read_csv(path_output, sep='|', on_bad_lines='skip')
    print(f"\n✅ UJI COBA SUKSES! Total Data Terkumpul: {df_hasil.shape[0]} baris.")
    display(df_hasil.head())
except Exception as e:
    print(f"❌ Gagal membaca CSV. Pemisah kolom mungkin berantakan. Error: {e}")

File berhasil disimpan di: ../../data/dataset_keluhan_ntg.csv

✅ UJI COBA SUKSES! Total Data Terkumpul: 45 baris.


,teks_keluhan_awam,teks_laporan_teknisi,kategori_aset,severity,root_cause,jenis_kerusakan,biaya_perbaikan
0,Wastafel Ina di Gedung B Lantai 4 Zona Selatan...,Teknisi menemukan kebocoran pada pipa wastafel...,Plumbing,Berat,Kebocoran,Kebocoran,200000
1,Wastafel di Gedung B Lantai 4 Zona Selatan tid...,Teknisi menemukan bahwa wastafel Ina memiliki ...,Plumbing,Sedang,Kebocoran,Kebocoran,150000
2,Ina wastafel bocor,Teknisi menemukan bahwa wastafel memiliki kebo...,Plumbing,Ringan,Kebocoran,Kebocoran,100000
3,Gedung B Lantai 4 Zona Selatan wastafel Ina rusak,Teknisi menemukan bahwa wastafel memiliki kebo...,Plumbing,Berat,Kebocoran,Kebocoran,250000
4,Wastafel Ina di Zona Selatan bocor,Teknisi menemukan bahwa wastafel memiliki kebo...,Plumbing,Sedang,Kebocoran,Kebocoran,180000


File berhasil disimpan di: ../../data/dataset_keluhan_ntg.csv

✅ UJI COBA SUKSES! Total Data Terkumpul: 45 baris.


,teks_keluhan_awam,teks_laporan_teknisi,kategori_aset,severity,root_cause,jenis_kerusakan,biaya_perbaikan
0,Wastafel Ina di Gedung B Lantai 4 Zona Selatan...,Teknisi menemukan kebocoran pada pipa wastafel...,Plumbing,Berat,Kebocoran,Kebocoran,200000
1,Wastafel di Gedung B Lantai 4 Zona Selatan tid...,Teknisi menemukan bahwa wastafel Ina memiliki ...,Plumbing,Sedang,Kebocoran,Kebocoran,150000
2,Ina wastafel bocor,Teknisi menemukan bahwa wastafel memiliki kebo...,Plumbing,Ringan,Kebocoran,Kebocoran,100000
3,Gedung B Lantai 4 Zona Selatan wastafel Ina rusak,Teknisi menemukan bahwa wastafel memiliki kebo...,Plumbing,Berat,Kebocoran,Kebocoran,250000
4,Wastafel Ina di Zona Selatan bocor,Teknisi menemukan bahwa wastafel memiliki kebo...,Plumbing,Sedang,Kebocoran,Kebocoran,180000
